# Per-version search metrics on two boards

V1-V6 measured once each on two boards — one quiet, one tactical, from
`tuning/harness.py`'s `documented_positions()` — at depth/floor 2: visited nodes,
elapsed time, score/move, reached depth where available, cutoff events, skipped
child branches and forward-pruned candidates. Each call runs in its own subprocess
with the version's weights pinned before import, guarded by a 10-second timeout.

In [1]:
import json
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

sys.dont_write_bytecode = True  # never write __pycache__ into the existing project tree


def find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "connect6" / "defines.py").is_file():
            return base.resolve()
    raise RuntimeError("run this notebook from inside the Connect6Engine repository")


ROOT = find_root()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "tuning"))
import harness  # reuse: documented_positions (read-only)

COMPILED_LIB = ROOT / "dist" / "lib"
for path in (COMPILED_LIB, ROOT / "tuning" / "default.json", ROOT / "tuning" / "winner.json"):
    assert path.exists(), f"missing prerequisite, experiment blocked: {path}"

WORKDIR = Path(tempfile.mkdtemp(prefix="assign_metrics_"))
NO_BOOK = WORKDIR / "book_disabled_nonexistent.json"  # disables book lookup

VERSIONS = [
    ("v1", "connect6.search.search_min_max", "default.json",
     "fixed-depth 2", "min-max, hand-tuned weights"),
    ("v2", "connect6.search.search_alpha_beta", "default.json",
     "fixed-depth 2", "alpha-beta, same hand-tuned weights"),
    ("v3", "connect6.search.search_alpha_beta", "winner.json",
     "fixed-depth 2", "alpha-beta, GA-champion weights"),
    ("v4", "connect6.search.search_alpha_beta_with_quiescence", "winner.json",
     "fixed-depth 2", "quiescence + forward pruning, champion weights"),
    ("v5", "connect6.vcf.search_v5", "winner.json",
     "budgeted: floor 2, V5_NODE_BUDGET=12000", "forced-win layer + deepening + defence screen"),
    ("v6", "connect6.parallel.search_v6", "winner.json",
     "budgeted fixed-mode: floor 2, V6_NODE_BUDGET=36000, V6_JOBS=1",
     "parallel split root (inline), PVS subtrees, defence screen"),
]

BOARDS = {}
for name, board, color, pre_move in harness.documented_positions():
    if name == "midgame, white to move":
        BOARDS["midgame (quiet)"] = (board, color, pre_move)
    elif name == "fork attack, black to move":
        BOARDS["fork attack (tactical)"] = (board, color, pre_move)
assert len(BOARDS) == 2, "expected the midgame and fork-attack boards"

DEPTH = 2
CALL_TIMEOUT = 10.0  # external timeout per measurement call, seconds


def worker_env(weights_file, pythonpath):
    return dict(
        os.environ,
        PYTHONPATH=str(pythonpath),
        WEIGHTS_FILE=str(ROOT / "tuning" / weights_file),
        BOOK_FILE=str(NO_BOOK),
        V6_JOBS="1",
        PYTHONDONTWRITEBYTECODE="1",
    )


def spec_for(version, board_name):
    board, color, pre_move = BOARDS[board_name]
    return {
        "callable": version[1],
        "board": [row[:] for row in board],
        "color": int(color),
        "pre_move": [[pre_move.positions[0].x, pre_move.positions[0].y],
                     [pre_move.positions[1].x, pre_move.positions[1].y]],
        "depth": DEPTH,
    }


def run_worker(source, spec, env, timeout):
    worker = WORKDIR / "worker.py"
    worker.write_text(source)
    try:
        done = subprocess.run(
            [sys.executable, str(worker)], input=json.dumps(spec), capture_output=True,
            text=True, timeout=timeout, env=env)
        if done.returncode != 0:
            return {"status": "error", "stderr_tail": done.stderr[-400:]}
        return {"status": "ok", **json.loads(done.stdout)}
    except subprocess.TimeoutExpired:
        return {"status": "timeout"}

print("setup ready: 6 versions x 2 boards =", 2 * len(VERSIONS), "measurement calls")

setup ready: 6 versions x 2 boards = 12 measurement calls


In [2]:
MEASURE_WORKER = r'''
import importlib
import json
import sys
import time

spec = json.loads(sys.stdin.read())
module_name, attr = spec["callable"].rsplit(".", 1)
fn = getattr(importlib.import_module(module_name), attr)
from connect6.defines import Color, Move, Position, SearchStats
from connect6.protocol import move2msg

board = spec["board"]
color = Color(spec["color"])
pre_move = Move((Position(spec["pre_move"][0][0], spec["pre_move"][0][1]),
                 Position(spec["pre_move"][1][0], spec["pre_move"][1][1])))
stats = SearchStats()
start = time.perf_counter()
result = fn(board, color, spec["depth"], pre_move, stats)
wall = time.perf_counter() - start
print(json.dumps({
    "score": result.score,
    "move_msg": move2msg(result.move),
    "move": [[p.x, p.y] for p in result.move.positions],
    "nodes": stats.node_count,
    "depth_reached": stats.depth_reached,   # only the v6 callable populates this
    "wall_s": round(wall, 4),
}))
'''

measurements = {}
for version in VERSIONS:
    for board_name in BOARDS:
        spec = spec_for(version, board_name)
        env = worker_env(version[2], COMPILED_LIB)
        row = run_worker(MEASURE_WORKER, spec, env, CALL_TIMEOUT)
        row["answer_kind"] = version[3]
        measurements[(version[0], board_name)] = row
        if row["status"] == "ok":
            shown = {k: row[k] for k in ("score", "move_msg", "nodes", "depth_reached", "wall_s")}
        else:
            shown = row.get("stderr_tail", row["status"].upper())
        print(f"{version[0]:<3} {board_name:<24} {row['status']:<8} {shown}", flush=True)

print(f"\n{sum(1 for r in measurements.values() if r['status'] == 'ok')}"
      f"/{len(measurements)} measurement calls completed within the {CALL_TIMEOUT:.0f}s timeout")

v1  midgame (quiet)          ok       {'score': -61, 'move_msg': 'FJII', 'nodes': 931, 'depth_reached': 0, 'wall_s': 0.0317}


v1  fork attack (tactical)   ok       {'score': 1025, 'move_msg': 'FJJN', 'nodes': 931, 'depth_reached': 0, 'wall_s': 0.0342}


v2  midgame (quiet)          ok       {'score': -61, 'move_msg': 'FJII', 'nodes': 212, 'depth_reached': 0, 'wall_s': 0.0137}


v2  fork attack (tactical)   ok       {'score': 1025, 'move_msg': 'FJJN', 'nodes': 175, 'depth_reached': 0, 'wall_s': 0.0134}


v3  midgame (quiet)          ok       {'score': -134, 'move_msg': 'FJII', 'nodes': 191, 'depth_reached': 0, 'wall_s': 0.0131}


v3  fork attack (tactical)   ok       {'score': 742, 'move_msg': 'FJJN', 'nodes': 173, 'depth_reached': 0, 'wall_s': 0.0136}


... (3 outputs left out)

v5  fork attack (tactical)   ok       {'score': 20000, 'move_msg': 'JSFJ', 'nodes': 18, 'depth_reached': 0, 'wall_s': 0.0322}


v6  midgame (quiet)          ok       {'score': -617, 'move_msg': 'JGII', 'nodes': 39524, 'depth_reached': 4, 'wall_s': 7.059}


v6  fork attack (tactical)   ok       {'score': 20000, 'move_msg': 'JSFJ', 'nodes': 18, 'depth_reached': 0, 'wall_s': 0.0314}



12/12 measurement calls completed within the 10s timeout


In [3]:
# Identity spot-check: the compiled dist package against the current interpreted source,
# same weights pinned, same callables, shallow cases.

identity_cases = [
    ("v2 fork d1 (default weights)", ("v2", "connect6.search.search_alpha_beta", "default.json"),
     "fork attack (tactical)", 1),
    ("v4 midgame d2 (champion weights)",
     ("v4", "connect6.search.search_alpha_beta_with_quiescence", "winner.json"),
     "midgame (quiet)", 2),
]

identity_ok = True
for label, version, board_name, depth in identity_cases:
    spec = spec_for(version, board_name)
    spec["depth"] = depth
    interp = run_worker(MEASURE_WORKER, spec, worker_env(version[2], ROOT), 60.0)
    comp = run_worker(MEASURE_WORKER, spec, worker_env(version[2], COMPILED_LIB), 60.0)
    keys = ("score", "move_msg", "nodes")
    same = (interp["status"] == "ok" and comp["status"] == "ok"
            and all(interp.get(k) == comp.get(k) for k in keys))
    identity_ok &= same
    print(f"{label:<36} interpreted={ {k: interp.get(k) for k in keys} } "
          f"compiled={ {k: comp.get(k) for k in keys} } -> {'IDENTICAL' if same else 'DIFFER'}")
if not identity_ok:
    print("WARNING - BLOCKED ASSUMPTION: the compiled package diverged from the interpreted "
          "source on a spot-check; all rows below are the compiled build's and the identity "
          "premise is NOT established for this run (see scripts/verify_compiled.py).")
else:
    print("\ncompiled package reproduces the source's answers and node counts on the spot-checks")

v2 fork d1 (default weights)         interpreted={'score': 10046, 'move_msg': 'JNJR', 'nodes': 31} compiled={'score': 10046, 'move_msg': 'JNJR', 'nodes': 31} -> IDENTICAL


v4 midgame d2 (champion weights)     interpreted={'score': -134, 'move_msg': 'FJII', 'nodes': 402} compiled={'score': -134, 'move_msg': 'FJII', 'nodes': 402} -> IDENTICAL

compiled package reproduces the source's answers and node counts on the spot-checks


In [4]:
# The notebook-local instrumented replica: it mirrors the production search loops and adds
# event counters; the building blocks are imported from the package. Every replica is
# calibrated against the original callable before its counts are used.

INSTRUMENT_WORKER = r'''
import json
import sys
import time

from connect6.board import copy_board, is_board_empty, make_move, unmake_move, opponent
from connect6.candidates import generate_candidate_moves, generate_scored_candidate_moves
from connect6.defines import (FORWARD_WIDTHS, MAXINT, MAX_CANDIDATE_MOVES, MININT,
                              QUIESCENCE_DEPTH, QUIESCENCE_MIN_THREATS, QUIESCENCE_WIDTH,
                              THREAT_WIN, V5_DEEPEN_STEPS, V5_NODE_BUDGET, V5_SCREEN_WIDTH,
                              V6_NODE_BUDGET, Color, Move, Position, SearchStats, SetWeight)
from connect6.evaluation import evaluate
from connect6.protocol import move2msg
from connect6.search import (search_alpha_beta, search_alpha_beta_with_quiescence,
                             search_min_max)
from connect6.vcf import find_forced_win, join_threats, search_v5
from connect6.parallel import search_v6

CENTER = Move((Position(10, 10), Position(10, 10)))


class Counters:
    def __init__(self):
        self.cutoffs = 0        # alpha-beta cutoff events in the main search
        self.skipped = 0        # immediate children not examined at cutoff nodes
        self.fwd_pruned = 0      # candidates discarded by FORWARD_WIDTHS width cuts
        self.q_cutoffs = 0       # cutoff events inside quiescence
        self.q_skipped = 0       # pool entries not examined at quiescence cutoffs
        self.q_pruned = 0        # candidates discarded by the quiescence width cut
        self.root_scores = []    # per-root-candidate scores (exact when prune is off)

    def as_dict(self):
        return {"cutoff_events": self.cutoffs,
                "skipped_child_branches": self.skipped,
                "forward_pruned_candidates": self.fwd_pruned,
                "quiescence_cutoff_events": self.q_cutoffs,
                "quiescence_skipped": self.q_skipped,
                "quiescence_width_pruned": self.q_pruned}


def _advance(maximizing, child_score, candidate, best_score, best_move, alpha, beta, prune):
    # mirrors connect6/search.py _advance
    if maximizing:
        if child_score > best_score:
            best_score = child_score
            best_move = candidate
        if prune:
            if best_score >= beta:
                return best_score, best_move, alpha, beta, True
            if best_score > alpha:
                alpha = best_score
    else:
        if child_score < best_score:
            best_score = child_score
            best_move = candidate
        if prune:
            if best_score <= alpha:
                return best_score, best_move, alpha, beta, True
            if best_score < beta:
                beta = best_score
    return best_score, best_move, alpha, beta, False


def r_alpha_beta(depth, board, color, pre_move, stats, alpha, beta, prune, enhanced, ply,
                 incoming, pvs, cnt):
    # mirrors connect6/search.py _alpha_beta, counters added
    if enhanced and depth <= 0:
        return r_quiescence(board, color, pre_move, stats, alpha, beta, incoming,
                            QUIESCENCE_DEPTH, cnt)
    stats.node_count += 1
    score = evaluate(board, pre_move)
    if score == MAXINT or score == MININT:
        return pre_move, score
    if depth <= 0:
        return pre_move, score
    threats = []
    if enhanced:
        width = FORWARD_WIDTHS[min(ply, len(FORWARD_WIDTHS) - 1)]
        full, full_threats = generate_scored_candidate_moves(board, MAX_CANDIDATE_MOVES, color)
        candidates = full[:width]
        threats = full_threats[:width]
        cnt.fwd_pruned += len(full) - len(candidates)
    else:
        candidates = generate_candidate_moves(board, MAX_CANDIDATE_MOVES, color)
    if len(candidates) == 0:
        return pre_move, score
    maximizing = color == Color.BLACK
    best_score = MININT if maximizing else MAXINT
    best_move = candidates[0]
    for index, candidate in enumerate(candidates):
        child_incoming = threats[index] if enhanced else 0
        make_move(board, candidate, color)
        if pvs and prune and index > 0 and alpha + 1 < beta:
            child = r_scouted_child(depth - 1, board, color, candidate, stats, alpha, beta,
                                    prune, enhanced, ply + 1, child_incoming, cnt)
        else:
            child = r_alpha_beta(depth - 1, board, opponent(color), candidate, stats,
                                 alpha, beta, prune, enhanced, ply + 1,
                                 child_incoming if child_incoming > 0 else 0, pvs, cnt)
        unmake_move(board, candidate)
        if ply == 0:
            cnt.root_scores.append(child[1])
        best_score, best_move, alpha, beta, cutoff = _advance(
            maximizing, child[1], candidate, best_score, best_move, alpha, beta, prune)
        if cutoff:
            cnt.cutoffs += 1
            cnt.skipped += len(candidates) - 1 - index
            return best_move, best_score
    return best_move, best_score


def r_scouted_child(depth, board, color, candidate, stats, alpha, beta, prune, enhanced,
                    ply, child_incoming, cnt):
    # mirrors connect6/search.py _scouted_child (PVS null-window scout + wide re-search)
    incoming = child_incoming if child_incoming > 0 else 0
    if color == Color.BLACK:
        scout = r_alpha_beta(depth, board, opponent(color), candidate, stats,
                             alpha, alpha + 1, prune, enhanced, ply, incoming, True, cnt)
        if scout[1] <= alpha:
            return scout
    else:
        scout = r_alpha_beta(depth, board, opponent(color), candidate, stats,
                             beta - 1, beta, prune, enhanced, ply, incoming, True, cnt)
        if scout[1] >= beta:
            return scout
    return r_alpha_beta(depth, board, opponent(color), candidate, stats,
                        alpha, beta, prune, enhanced, ply, incoming, True, cnt)


def r_quiescence(board, color, pre_move, stats, alpha, beta, incoming, qdepth, cnt):
    # mirrors connect6/search.py _quiescence, counters added
    stats.node_count += 1
    maximizing = color == Color.BLACK
    stand = evaluate(board, pre_move)
    if stand == MAXINT or stand == MININT:
        return pre_move, stand
    loud = incoming > 0 or abs(stand) >= int(SetWeight.LIVE_FIVE)
    if not loud:
        if maximizing and stand >= beta:
            return pre_move, stand
        if not maximizing and stand <= alpha:
            return pre_move, stand
    if qdepth <= 0:
        return pre_move, stand
    full, full_threats = generate_scored_candidate_moves(board, MAX_CANDIDATE_MOVES, color)
    moves = full[:QUIESCENCE_WIDTH]
    threats = full_threats[:QUIESCENCE_WIDTH]
    cnt.q_pruned += len(full) - len(moves)
    if len(moves) == 0:
        return pre_move, stand
    for move, threat in zip(moves, threats):
        if threat >= THREAT_WIN:
            return move, MAXINT if maximizing else MININT
    if loud:
        pool = list(zip(moves, threats))
        best_score = MININT if maximizing else MAXINT
        best_move = moves[0]
    else:
        pool = [(move, threat) for move, threat in zip(moves, threats)
                if threat >= QUIESCENCE_MIN_THREATS]
        if len(pool) == 0:
            return pre_move, stand
        best_score = stand
        best_move = pre_move
        if maximizing and stand > alpha:
            alpha = stand
        if not maximizing and stand < beta:
            beta = stand
    for index, (move, threat) in enumerate(pool):
        make_move(board, move, color)
        child = r_quiescence(board, opponent(color), move, stats, alpha, beta,
                             threat if threat > 0 else 0, qdepth - 1, cnt)
        unmake_move(board, move)
        best_score, best_move, alpha, beta, cutoff = _advance(
            maximizing, child[1], move, best_score, best_move, alpha, beta, True)
        if cutoff:
            cnt.q_cutoffs += 1
            cnt.q_skipped += len(pool) - 1 - index
            return best_move, best_score
    return best_move, best_score


def replica_single(kind, board, color, depth, pre_move, cnt):
    # mirrors connect6/search.py _search for the v1-v4 callables
    stats = SearchStats()
    working = copy_board(board)
    if is_board_empty(working):
        return stats, CENTER, 0
    prune = kind != "v1"
    enhanced = kind == "v4"
    move, score = r_alpha_beta(depth, working, color, pre_move, stats,
                               MININT, MAXINT, prune, enhanced, 0, 0, False, cnt)
    return stats, move, score


def replica_v5(board, color, depth, pre_move, cnt):
    # mirrors connect6/vcf.py search_v5 (stage 1 + deepening + screen fallback)
    stats = SearchStats()
    if is_board_empty(board):
        return stats, CENTER, 0, 0
    win = find_forced_win(board, color, stats)
    if win is not None:
        return stats, win, (MAXINT if color == Color.BLACK else MININT), 0
    best_move, best_score = None, 0
    spent = 0
    reached = depth
    for step in range(V5_DEEPEN_STEPS + 1):
        iteration = SearchStats()
        working = copy_board(board)
        best_move, best_score = r_alpha_beta(depth + step, working, color, pre_move,
                                             iteration, MININT, MAXINT, True, True, 0, 0,
                                             False, cnt)
        spent += iteration.node_count
        stats.node_count += iteration.node_count
        reached = depth + step
        if spent >= V5_NODE_BUDGET or best_score in (MAXINT, MININT):
            break
    if best_score in (MAXINT, MININT):
        return stats, best_move, best_score, reached
    other = opponent(color)
    work = copy_board(board)
    make_move(work, best_move, color)
    loses = find_forced_win(work, other, stats) is not None
    armed = len(join_threats(work, other, stats))
    if not loses and armed == 0:
        return stats, best_move, best_score, reached
    quality = (1 if loses else 0, armed)
    candidates, _ = generate_scored_candidate_moves(board, FORWARD_WIDTHS[0], color)
    pool = [move for move in candidates[:V5_SCREEN_WIDTH] if move != best_move]
    work2 = copy_board(board)
    make_move(work2, best_move, color)
    cells = set()
    for join in join_threats(work2, other, stats):
        cells.update(((join.positions[0].x, join.positions[0].y),
                      (join.positions[1].x, join.positions[1].y)))
    for first in sorted(cells):
        for second in sorted(cells):
            if first < second:
                pool.append(Move((Position(first[0], first[1]), Position(second[0], second[1]))))
    chosen_move, chosen_score = best_move, best_score
    for candidate in pool:
        probe = copy_board(board)
        make_move(probe, candidate, color)
        probe_loses = find_forced_win(probe, other, stats) is not None
        probe_armed = len(join_threats(probe, other, stats))
        if (1 if probe_loses else 0, probe_armed) < quality:
            quality = (1 if probe_loses else 0, probe_armed)
            chosen_move = candidate
    return stats, chosen_move, chosen_score, reached


def replica_subtree(board, color, depth, candidate, incoming, cnt):
    # mirrors connect6/search.py search_candidate_subtree (one split-root job)
    stats = SearchStats()
    work = copy_board(board)
    make_move(work, candidate, color)
    _move, score = r_alpha_beta(depth - 1, work, opponent(color), candidate, stats,
                                MININT, MAXINT, True, True, 1,
                                incoming if incoming > 0 else 0, True, cnt)
    return score, stats.node_count


def replica_split_root(board, color, depth, pre_move, cnt, stats):
    # mirrors connect6/parallel.py _split_root on the inline V6_JOBS=1 path
    candidates, threats = generate_scored_candidate_moves(board, FORWARD_WIDTHS[0], color)
    if not candidates:
        stats.node_count += 1
        return pre_move, evaluate(board, pre_move)
    maximizing = color == Color.BLACK
    best_score = MININT if maximizing else MAXINT
    best_move = candidates[0]
    nodes = 1
    for index, candidate in enumerate(candidates):
        score, spent = replica_subtree(board, color, depth, candidate, threats[index], cnt)
        nodes += spent
        if maximizing:
            if score > best_score:
                best_score, best_move = score, candidate
        elif score < best_score:
            best_score, best_move = score, candidate
        if best_score == (MAXINT if maximizing else MININT):
            break
    stats.node_count += nodes
    return best_move, best_score


def replica_screen(board, color, best_move, best_score, stats):
    # mirrors connect6/parallel.py _screen with the inline _probe_defenses path
    other = opponent(color)
    work = copy_board(board)
    make_move(work, best_move, color)
    loses = find_forced_win(work, other, stats) is not None
    armed = join_threats(work, other, stats)
    quality = (1 if loses else 0, len(armed))
    if quality == (0, 0):
        return best_move, best_score
    candidates, _ = generate_scored_candidate_moves(board, FORWARD_WIDTHS[0], color)
    pool = [move for move in candidates[:V5_SCREEN_WIDTH] if move != best_move]
    cells = set()
    for join in armed:
        cells.update(((join.positions[0].x, join.positions[0].y),
                      (join.positions[1].x, join.positions[1].y)))
    for first in sorted(cells):
        for second in sorted(cells):
            if first < second:
                pool.append(Move((Position(first[0], first[1]), Position(second[0], second[1]))))
    probed = []
    for candidate in pool:
        probe = copy_board(board)
        make_move(probe, candidate, color)
        probe_loses = 1 if find_forced_win(probe, other, stats) is not None else 0
        probe_armed = len(join_threats(probe, other, stats))
        probed.append((probe_loses, probe_armed))
    chosen_move = best_move
    for candidate, quality_probed in zip(pool, probed):
        if quality_probed < quality:
            quality = quality_probed
            chosen_move = candidate
    return chosen_move, best_score


def replica_v6(board, color, depth, pre_move, cnt):
    # mirrors connect6/parallel.py search_v6 in fixed mode (budget=None):
    # stage 1 + _fixed_deepen + _screen, V6_JOBS=1 keeps every pool path inline
    stats = SearchStats()
    if is_board_empty(board):
        return stats, CENTER, 0, 0
    win = find_forced_win(board, color, stats)
    if win is not None:
        return stats, win, (MAXINT if color == Color.BLACK else MININT), 0
    best_move, best_score = None, 0
    spent = 0
    reached = depth
    for step in range(V5_DEEPEN_STEPS + 1):
        iteration = SearchStats()
        best_move, best_score = replica_split_root(board, color, depth + step, pre_move,
                                                   cnt, iteration)
        spent += iteration.node_count
        stats.node_count += iteration.node_count
        reached = depth + step
        if spent >= V6_NODE_BUDGET or best_score in (MAXINT, MININT):
            break
    if best_score in (MAXINT, MININT):
        return stats, best_move, best_score, reached
    chosen_move, chosen_score = replica_screen(board, color, best_move, best_score, stats)
    return stats, chosen_move, chosen_score, reached


ORIGINALS = {
    "v1": search_min_max,
    "v2": search_alpha_beta,
    "v3": search_alpha_beta,
    "v4": search_alpha_beta_with_quiescence,
    "v5": search_v5,
    "v6": search_v6,
}


def run_case(version, board, color, depth, pre_move):
    original_stats = SearchStats()
    start = time.perf_counter()
    original = ORIGINALS[version](board, color, depth, pre_move, original_stats)
    wall_original = time.perf_counter() - start

    cnt = Counters()
    start = time.perf_counter()
    if version in ("v1", "v2", "v3", "v4"):
        rep_stats, rep_move, rep_score = replica_single(version, board, color, depth,
                                                       pre_move, cnt)
        rep_reached = None
    elif version == "v5":
        rep_stats, rep_move, rep_score, rep_reached = replica_v5(board, color, depth,
                                                                 pre_move, cnt)
    else:
        rep_stats, rep_move, rep_score, rep_reached = replica_v6(board, color, depth,
                                                                pre_move, cnt)
    wall_replica = time.perf_counter() - start

    equal = (original.score == rep_score and original.move == rep_move
             and original_stats.node_count == rep_stats.node_count)
    if version == "v6":
        equal = equal and original_stats.depth_reached == rep_reached
    root_ties = None
    if version == "v1":
        root_ties = sum(1 for s in cnt.root_scores if s == rep_score)
    return {
        "original": {"score": original.score, "move_msg": move2msg(original.move),
                    "nodes": original_stats.node_count,
                    "depth_reached": original_stats.depth_reached},
        "replica": {"score": rep_score, "move_msg": move2msg(rep_move),
                    "nodes": rep_stats.node_count},
        "equal": equal,
        "replica_reached_depth": rep_reached,
        "root_ties": root_ties,
        "counters": cnt.as_dict(),
        "wall_original_s": round(wall_original, 4),
        "wall_replica_s": round(wall_replica, 4),
    }


spec = json.loads(sys.stdin.read())
result = run_case(spec["version"], spec["board"], Color(spec["color"]), spec["depth"],
                  Move((Position(spec["pre_move"][0][0], spec["pre_move"][0][1]),
                       Position(spec["pre_move"][1][0], spec["pre_move"][1][1]))))
print(json.dumps(result))
'''

print(f"instrumented replica source: {len(INSTRUMENT_WORKER.splitlines())} notebook-local lines")

instrumented replica source: 405 notebook-local lines


In [5]:
# Calibration case per version, at depth/floor 1.
CALIB = {"v1": "fork attack (tactical)", "v2": "fork attack (tactical)",
         "v3": "fork attack (tactical)", "v4": "fork attack (tactical)",
         "v5": "midgame (quiet)", "v6": "midgame (quiet)"}

instrumented = {}
for version in VERSIONS:
    calib_name = CALIB[version[0]]
    calib_spec = spec_for(version, calib_name)
    calib_spec["depth"] = 1
    calib_spec["version"] = version[0]
    env = worker_env(version[2], COMPILED_LIB)
    calib = run_worker(INSTRUMENT_WORKER, calib_spec, env, 120.0)
    instrumented[("calib", version[0])] = calib
    print(f"{version[0]} calibration on {calib_name} at floor/depth 1: "
          f"{calib.get('status')} equal={calib.get('equal')}", flush=True)
    for board_name in BOARDS:
        spec = spec_for(version, board_name)
        spec["depth"] = DEPTH
        spec["version"] = version[0]
        row = run_worker(INSTRUMENT_WORKER, spec, env, 300.0)
        instrumented[(version[0], board_name)] = row
        print(f"{version[0]:<3} {board_name:<24} instrumented: {row.get('status')} "
              f"equal={row.get('equal')}", flush=True)

v1 calibration on fork attack (tactical) at floor/depth 1: ok equal=True


v1  midgame (quiet)          instrumented: ok equal=True


v1  fork attack (tactical)   instrumented: ok equal=True


v2 calibration on fork attack (tactical) at floor/depth 1: ok equal=True


v2  midgame (quiet)          instrumented: ok equal=True


v2  fork attack (tactical)   instrumented: ok equal=True


... (8 outputs left out)

v5  fork attack (tactical)   instrumented: ok equal=True


v6 calibration on midgame (quiet) at floor/depth 1: ok equal=True


v6  midgame (quiet)          instrumented: ok equal=True


v6  fork attack (tactical)   instrumented: ok equal=True


In [6]:
def reached_label(version_id, measurement, instrumented_row):
    if version_id in ("v1", "v2", "v3", "v4"):
        return "2 (fixed-depth: the whole depth-2 tree is searched by construction)"
    if version_id == "v5":
        if instrumented_row.get("equal"):
            return (f"{instrumented_row['replica_reached_depth']} "
                    "(replica-derived; the v5 callable does not record reached depth)")
        return "unavailable"
    got = measurement.get("depth_reached")
    if got is None:
        return "unavailable"
    extra = ""
    if instrumented_row.get("equal"):
        extra = f" (replica cross-check: {instrumented_row['replica_reached_depth']})"
    return f"{got}{extra}"


print(f"{'ver':<3} {'board':<22} {'answer kind':<38} {'nodes':>7} {'wall s':>7} "
      f"{'score':>7} {'move':<6} {'reached depth'}")
for version in VERSIONS:
    for board_name in BOARDS:
        key = (version[0], board_name)
        meas = measurements[key]
        instr = instrumented.get(key, {})
        if meas["status"] != "ok":
            print(f"{version[0]:<3} {board_name:<22} {version[3]:<38} "
                  f"{'-':>7} {'-':>7} {'-':>7} {'-':<6} call {meas['status']}")
            continue
        print(f"{version[0]:<3} {board_name:<22} {version[3]:<38} {meas['nodes']:>7} "
              f"{meas['wall_s']:>7.2f} {meas['score']:>7} {meas['move_msg']:<6} "
              f"{reached_label(version[0], meas, instr)}")

print("\npruned-node decomposition (from the calibrated instrumented replica; "
      "main-search metrics first, quiescence's own cuts separately)")
print(f"{'ver':<3} {'board':<22} {'calib':<6} {'match':<6} {'cutoffs':>8} {'skipped':>8} "
      f"{'fwd-pruned':>10} {'q-cutoffs':>9} {'q-skipped':>9} {'q-pruned':>9} {'root ties':>9}")
for version in VERSIONS:
    for board_name in BOARDS:
        key = (version[0], board_name)
        meas, instr = measurements[key], instrumented.get(key, {})
        calib_ok = instrumented.get(("calib", version[0]), {}).get("equal")
        if instr.get("status") != "ok" or not instr.get("equal") or not calib_ok:
            print(f"{version[0]:<3} {board_name:<22} {str(calib_ok):<6} {'no':<6} "
                  f"{'unavailable':>8}")
            continue
        c = instr["counters"]
        ties = instr.get("root_ties")
        ties_shown = str(ties) if ties is not None else "-"
        print(f"{version[0]:<3} {board_name:<22} {'yes':<6} {'yes':<6} {c['cutoff_events']:>8} "
              f"{c['skipped_child_branches']:>8} {c['forward_pruned_candidates']:>10} "
              f"{c['quiescence_cutoff_events']:>9} {c['quiescence_skipped']:>9} "
              f"{c['quiescence_width_pruned']:>9} {ties_shown:>9}")

ver board                  answer kind                              nodes  wall s   score move   reached depth
v1  midgame (quiet)        fixed-depth 2                              931    0.03     -61 FJII   2 (fixed-depth: the whole depth-2 tree is searched by construction)
v1  fork attack (tactical) fixed-depth 2                              931    0.03    1025 FJJN   2 (fixed-depth: the whole depth-2 tree is searched by construction)
v2  midgame (quiet)        fixed-depth 2                              212    0.01     -61 FJII   2 (fixed-depth: the whole depth-2 tree is searched by construction)
v2  fork attack (tactical) fixed-depth 2                              175    0.01    1025 FJJN   2 (fixed-depth: the whole depth-2 tree is searched by construction)
v3  midgame (quiet)        fixed-depth 2                              191    0.01    -134 FJII   2 (fixed-depth: the whole depth-2 tree is searched by construction)
v3  fork attack (tactical) fixed-depth 2                        

In [7]:
# V1/V2 check: equal scores on the same candidate tree, and an explanation of ties.

for board_name in BOARDS:
    v1 = measurements[("v1", board_name)]
    v2 = measurements[("v2", board_name)]
    if v1["status"] != "ok" or v2["status"] != "ok":
        print(f"{board_name}: v1/v2 rows missing ({v1['status']}/{v2['status']}) - check blocked")
        continue
    same_score = v1["score"] == v2["score"]
    same_move = v1["move_msg"] == v2["move_msg"]
    instr_v1 = instrumented[("v1", board_name)]
    ties = instr_v1.get("root_ties") if instr_v1.get("equal") else None
    print(f"{board_name}: v1 score={v1['score']} move={v1['move_msg']} nodes={v1['nodes']}")
    print(f"{'':>22} v2 score={v2['score']} move={v2['move_msg']} nodes={v2['nodes']}")
    print(f"{'':>22} scores equal: {same_score}, moves equal: {same_move}, "
          f"root candidates tying at the best score (v1's exact tree): {ties}")
    if not same_score:
        print(f"{'':>22} RECORDING A MISMATCH: v1/v2 scores differ on the same candidate tree")
    if not same_move:
        print(f"{'':>22} RECORDING A MISMATCH: v1/v2 moves differ on the same candidate tree")

print("\nExplanation. Both variants update the running best only on a strictly better score, "
      "so the chosen move is the first root candidate, in generation order, that reaches the "
      "minimax value. Alpha-beta cannot cut off at the root (its window is the full "
      "(MININT, MAXINT)), and a cut-off child returns a bound that never beats the running "
      "best, so v2's root choice is identical to v1's - alpha-beta changes the node count, "
      "never the answer. When several root candidates tie at the best score, the reported move "
      "is simply the earliest in candidate order (completion pairs first, then the "
      "corrected-sum + threat rank): the tie is broken by the generator's ordering, not by the "
      "search. Alpha-beta cannot count those exact ties among pruned siblings (a cut-off child "
      "returns a bound, not its true value), which is why the tie count comes from v1's "
      "full-width replica.")

midgame (quiet): v1 score=-61 move=FJII nodes=931
                       v2 score=-61 move=FJII nodes=212
                       scores equal: True, moves equal: True, root candidates tying at the best score (v1's exact tree): 2
fork attack (tactical): v1 score=1025 move=FJJN nodes=931
                       v2 score=1025 move=FJJN nodes=175
                       scores equal: True, moves equal: True, root candidates tying at the best score (v1's exact tree): 1

Explanation. Alpha-beta changes the node count, never the answer; ties at the best score are broken by the generator's ordering.


## Notes

Workers run in a fresh temporary directory with weights pinned before import, so
no project file is touched.

Reproduction: `uv run python scripts/nbclient_run.py benchmarks/assignment_coverage/02_search_metrics.ipynb`